In [5]:
!pip install numpy
!pip install pandas
!pip install matplotlib
!pip install scikit-learn
!pip install joblib
!pip install jupyter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 13.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 13.8 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 13.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 14.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 14.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43/43 [jupyter]1/43 [notebook]b]ver]]t]


In [9]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [10]:
DATA_PATH = "/home/ash/Projects/BMS/SpamProbabilityEngine/data/sms+spam+collection/SMSSpamCollection"
df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    header=None,
    names=["label","message"]
)

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [11]:
print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [12]:
print("Number of messages:", len(df))

print("\nLabels:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Number of messages: 5572

Labels:
label
ham     4825
spam     747
Name: count, dtype: int64

Missing values:
label      0
message    0
dtype: int64


In [13]:
df["target"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

df.head()

,label,message,target
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [14]:
X = df["message"]
y = df["target"]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [16]:
vectorizer = CountVectorizer()

X_train_vectorized = vectorizer.fit_transform(X_train)

X_test_vectorized = vectorizer.transform(X_test)

In [19]:

print(vectorizer.get_feature_names_out()[:30])

['00' '000' '000pes' '008704050406' '0089' '0121' '01223585236'
 '01223585334' '02' '0207' '02072069400' '02073162414' '02085076972' '021'
 '03' '04' '0430' '05' '050703' '0578' '06' '07' '07008009200'
 '07046744435' '07090298926' '07099833605' '07123456789' '0721072'
 '07732584351' '07734396839']


In [20]:
model = MultinomialNB()

model.fit(
    X_train_vectorized,
    y_train
)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [24]:
message = ["Congratulations! You won an iphone!"]

message_vector = vectorizer.transform(message)

prediction = model.predict(message_vector)

print(prediction)

[1]


In [25]:
probability = model.predict_proba(message_vector)

print(probability)

[[0.06689555 0.93310445]]


In [26]:
print(
    f"Spam probability: {probability[0][1]:.2%}"
)

Spam probability: 93.31%


In [36]:
def check_message(message):

    message_vector = vectorizer.transform([message])

    prediction = model.predict(message_vector)[0]

    probability = model.predict_proba(
        message_vector
    )[0][1]

    if prediction == 1:
        result = "SPAM"
    else:
        result = "NOT SPAM"

    print("Message:", message)
    print("Prediction:", result)
    print(
        f"Spam probability: {probability:.2%}"
    )

In [40]:
message = input("Enter a message: ")

check_message(message)

Message: Where are you?
Prediction: NOT SPAM
Spam probability: 0.27%


In [41]:
y_pred = model.predict(
    X_test_vectorized
)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    f"Accuracy: {accuracy:.2%}"
)

Accuracy: 99.19%


In [42]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Ham", "Spam"]
    )
)

              precision    recall  f1-score   support

         Ham       0.99      1.00      1.00       966
        Spam       1.00      0.94      0.97       149

    accuracy                           0.99      1115
   macro avg       1.00      0.97      0.98      1115
weighted avg       0.99      0.99      0.99      1115



In [43]:
messages = [
    "Congratulations! You won a free prize!",
    "Hey, are you coming to class tomorrow?",
    "URGENT! Claim your reward now!",
    "Can you send me the project file?",
    "FREE money waiting for you!"
]

for message in messages:

    check_message(message)
    print("-" * 50)

Message: Congratulations! You won a free prize!
Prediction: SPAM
Spam probability: 100.00%
--------------------------------------------------
Message: Hey, are you coming to class tomorrow?
Prediction: NOT SPAM
Spam probability: 0.00%
--------------------------------------------------
Message: URGENT! Claim your reward now!
Prediction: SPAM
Spam probability: 100.00%
--------------------------------------------------
Message: Can you send me the project file?
Prediction: NOT SPAM
Spam probability: 0.03%
--------------------------------------------------
Message: FREE money waiting for you!
Prediction: NOT SPAM
Spam probability: 16.04%
--------------------------------------------------
